In [5]:
%load_ext autoreload
%autoreload 2

import yaml
from balance_metrics import plot_epoch_metrics_vs_accuracies, get_data, plot_layer_metrics, plot_exp_regressions, plot_linear_regressions

epoch_path = "../../experiment_data/balance_metrics/cross_epoch.csv"
layer_path = "../../experiment_data/balance_metrics/cross_layer.csv"
block_path = "../../experiment_data/balance_metrics/intra_block.csv"

epoch_better_path = "../../experiment_data/balance_metrics/cross_epoch_better.csv"

epoch_data = get_data(epoch_path)
layer_data = get_data(layer_path)
block_data = get_data(block_path)
epoch_data_better = get_data(epoch_better_path)

with open('../layer_orders_cross_layer.yml', 'r') as f:
    layer_order = yaml.safe_load(f)

with open('../layer_orders_intra_block.yml', 'r') as f:
    block_order = yaml.safe_load(f)
    
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

epoch_data_better["model"] = epoch_data_better["model"].apply(lambda x: f"{x}_better")
epoch_data_better.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,dataset,split,model,k,epoch,layer,thresh,thresh_mode,node_count,average_branching_factor,...,average_colless_index,total_colless,missing_colless,missing_colless_frac,sackin_index,average_sackin_index,leaf_count,total_volume,train_acc,val_acc
0,cifar,train,densenet_better,20,0,a1,0.0,0.0,2294,2.099817,...,5356.109658,994,98,0.089744,52616904.0,43774.462562,1202,50000,0.09992,0.0998
1,cifar,train,densenet_better,20,125,a1,0.0,0.0,710,2.055072,...,653.790123,324,21,0.060870,2991251.0,8195.208219,365,50000,0.99836,0.8659
2,cifar,train,densenet_better,20,150,a1,0.0,0.0,868,2.069212,...,680.676923,390,29,0.069212,2190354.0,4878.293987,449,50000,0.99990,0.8752
3,cifar,train,densenet_better,20,176,a1,0.0,0.0,306,2.046980,...,851.496454,141,8,0.053691,3013258.0,19192.726115,157,50000,0.97020,0.8464
4,cifar,train,densenet_better,20,200,a1,0.0,0.0,396,2.046632,...,960.269231,182,11,0.056995,2684267.0,13222.990148,203,50000,0.97902,0.8529


In [6]:
epoch_trainUval = pd.concat([epoch_data[(epoch_data['split'] == 'trainUval')], epoch_data_better[(epoch_data_better['split'] == 'trainUval')]])
epoch_trainUval = epoch_trainUval.sort_values(by='epoch')

fig = make_subplots(specs=[[{"secondary_y": True}]], )

colless_scatter_trainUval = px.scatter(epoch_trainUval, x='epoch', y='colless_index', color="dataset", color_discrete_sequence=color_seq, symbol="model")
train_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='train_acc', color="dataset", color_discrete_sequence=color_seq_train, symbol="model")
val_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='val_acc', color="dataset", color_discrete_sequence=color_seq_val, symbol="model", line_dash_sequence=['dash'])

for i in range(16):
	# get dataset, model out of the trace
	dataset, model = str(colless_scatter_trainUval.data[i]['name']).split(", ")
    
	all_epochs = epoch_trainUval[(epoch_trainUval['dataset'] == dataset) & (epoch_trainUval['model'] == model)]
	# find the epoch where val acc is maximum
	best_epoch = all_epochs.sort_values(by='val_acc', ascending=False).iloc[0]['epoch']

	fig.add_trace(
		colless_scatter_trainUval.data[i],
		secondary_y=False,
	)

	fig.update_yaxes(type="log", secondary_y=False)

	fig.add_trace(
		train_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	fig.add_trace(
		val_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	# add a vertical line at best epoch
	fig.add_trace(go.Scatter(x=[best_epoch, best_epoch], y=[0, 1], mode="lines", line=dict(color=colless_scatter_trainUval.data[i]['marker']['color'], dash="dot"), legendgroup=colless_scatter_trainUval.data[i]['name'], showlegend=False), secondary_y=True)

# set title
fig.update_layout(
	title_text="Cross-Epoch on TrainUVal: Colless Index vs Train/Val Accuracy"
)
fig.show()

In [7]:
epoch_trainUval = pd.concat([epoch_data[(epoch_data['split'] == 'trainUval')], epoch_data_better[(epoch_data_better['split'] == 'trainUval')]])
epoch_trainUval = epoch_trainUval.sort_values(by='epoch')

fig = make_subplots(specs=[[{"secondary_y": True}]], )

sackin_scatter_trainUval = px.scatter(epoch_trainUval, x='epoch', y='sackin_index', color="dataset", color_discrete_sequence=color_seq, symbol="model")
train_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='train_acc', color="dataset", color_discrete_sequence=color_seq_train, symbol="model")
val_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='val_acc', color="dataset", color_discrete_sequence=color_seq_val, symbol="model", line_dash_sequence=['dash'])

for i in range(16):
	# get dataset, model out of the trace
	dataset, model = str(sackin_scatter_trainUval.data[i]['name']).split(", ")
    
	all_epochs = epoch_trainUval[(epoch_trainUval['dataset'] == dataset) & (epoch_trainUval['model'] == model)]
	# find the epoch where val acc is maximum
	best_epoch = all_epochs.sort_values(by='val_acc', ascending=False).iloc[0]['epoch']

	fig.add_trace(
		sackin_scatter_trainUval.data[i],
		secondary_y=False,
	)

	fig.update_yaxes(type="log", secondary_y=False)

	fig.add_trace(
		train_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	fig.add_trace(
		val_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	# add a vertical line at best epoch
	fig.add_trace(go.Scatter(x=[best_epoch, best_epoch], y=[0, 1], mode="lines", line=dict(color=sackin_scatter_trainUval.data[i]['marker']['color'], dash="dot"), legendgroup=sackin_scatter_trainUval.data[i]['name'], showlegend=False), secondary_y=True)

# set title
fig.update_layout(
	title_text="Cross Epoch on TrainUVal: Sackin Index vs Train/Val Accuracy"
)
fig.show()